# Reproduzir e depurar falhas

Reduza um problema a um registro sem ID e faça a leitura falhar de propósito. Antes de executar, preveja a classe. Retome [dicionários](https://devrgs.github.io/Aprendendo-linguagens/#/python/registros/dicionarios), [exceções](https://devrgs.github.io/Aprendendo-linguagens/#/python/arquivos/excecoes) e [testes](https://devrgs.github.io/Aprendendo-linguagens/#/python/qualidade/testes-unitarios).

Primeiro reproduza o problema com os menores dados que ainda causam a falha. Inspecione as chaves, os valores e seus tipos. A última linha do traceback identifica a exceção. As linhas acima indicam por onde a chamada passou. Registre um caso de regressão ao corrigir, para detectar se o problema voltar. Em um filtro de faltantes, compare com `None` explicitamente para manter o zero.

E1 e E2 capturam as falhas intencionais. No caderno, teste uma chamada sem captura em célula separada para obter o traceback completo. No terminal, faça isso em uma cópia descartável. Em E3, qual teste prova que o código continua descartando faltantes? Em seguida, veja [custo de operações](https://devrgs.github.io/Aprendendo-linguagens/#/python/qualidade/custo-de-operacoes).

Execute as células na ordem com **Shift+Enter**. A preparação cria uma pasta nova na área de arquivos do caderno. Baixe o caderno editado e os resultados que quiser guardar.


In [ ]:
from pathlib import Path as _Path
from tempfile import mkdtemp as _mkdtemp
import os as _os
import sys as _sys
from importlib import invalidate_caches as _refresh_imports

# Auxiliares editáveis devem ser relidos sem aproveitar bytecode antigo.
_sys.dont_write_bytecode = True

_raiz = _Path.cwd()
_trabalho = _Path(_mkdtemp(prefix='python-depuracao-', dir=_raiz))


## E1 · Preveja e execute

Identifique a chave que causa KeyError em um registro contendo apenas valor. Inspecione as chaves antes do acesso.


In [ ]:
_pasta = _trabalho / 'e1'
_pasta.mkdir(exist_ok=True)
_os.chdir(_pasta)
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]
if str(_pasta) not in _sys.path:
    _sys.path.insert(0, str(_pasta))
_refresh_imports()


In [ ]:
registro = {"valor": "7"}
print(sorted(registro))
try:
    print(registro["id"])
except KeyError as erro:
    print(type(erro).__name__, erro.args[0])


In [ ]:
# Voltar à pasta do caderno para a etapa seguinte.
_os.chdir(_raiz)
if str(_pasta) in _sys.path:
    _sys.path.remove(str(_pasta))
# Liberar módulos locais para a próxima etapa importar sua própria versão.
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]


Saída esperada

```text
['valor']
KeyError id
```

A lista de chaves não contém id. A falha intencional ocorre no acesso com colchetes, antes de qualquer cálculo.


## E2 · Altere ou depure

Compare um registro sem valor e outro com valor "sete". Preveja se falham no mesmo ponto.


In [ ]:
_pasta = _trabalho / 'e2'
_pasta.mkdir(exist_ok=True)
_os.chdir(_pasta)
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]
if str(_pasta) not in _sys.path:
    _sys.path.insert(0, str(_pasta))
_refresh_imports()


In [ ]:
registros = [{"id": "a"}, {"id": "b", "valor": "sete"}]
for registro in registros:
    try:
        texto = registro["valor"]
        valor = int(texto)
    except KeyError:
        print(registro["id"], "campo ausente")
    except ValueError:
        print(registro["id"], "valor inválido")


In [ ]:
# Voltar à pasta do caderno para a etapa seguinte.
_os.chdir(_raiz)
if str(_pasta) in _sys.path:
    _sys.path.remove(str(_pasta))
# Liberar módulos locais para a próxima etapa importar sua própria versão.
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]


Saída esperada

```text
a campo ausente
b valor inválido
```

No primeiro registro, ler pela chave falha. No segundo, a chave existe e a conversão falha. O erro indica qual operação investigar.


## E3 · Resolva

Corrija o filtro que descarta zero junto com valores ausentes. O contrato considera somente None como ausente. Preserve a ordem, faça [None, 0, 5] virar [0, 5] e [] virar []. Inclua um teste de regressão para o zero.


In [ ]:
_pasta = _trabalho / 'e3'
_pasta.mkdir(exist_ok=True)
_os.chdir(_pasta)
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]
if str(_pasta) not in _sys.path:
    _sys.path.insert(0, str(_pasta))
_refresh_imports()


In [ ]:
def antes(medidas):
    return [n for n in medidas if n]

def sem_ausentes(medidas):
    return [n for n in medidas if n is not None]

entrada = [None, 0, 5]
print(antes(entrada))
print(sem_ausentes(entrada))
assert sem_ausentes(entrada) == [0, 5]
assert sem_ausentes([]) == []
assert sem_ausentes([None]) == []
print("regressão verificada")


In [ ]:
# Voltar à pasta do caderno para a etapa seguinte.
_os.chdir(_raiz)
if str(_pasta) in _sys.path:
    _sys.path.remove(str(_pasta))
# Liberar módulos locais para a próxima etapa importar sua própria versão.
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]


Saída esperada

```text
[5]
[0, 5]
regressão verificada
```

if n trata zero e None como falsos. A comparação de identidade retira apenas None. O teste que exige zero seria reprovado pelo filtro antigo, e o caso com None sozinho confirma a remoção.
